<a href="https://colab.research.google.com/github/TamirPalay/DI_Exercises/blob/main/week14/day3-4/Lesson1_Working_with_vector_databases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import BertTokenizer, BertModel
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [ ]:
import torch

# Define the input text
text = "PyTorch makes working with deep learning models intuitive and flexible."

# Tokenize input text and convert it to PyTorch tensors
inputs = tokenizer(text, return_tensors='pt')

# Inference mode: gradients are not computed
with torch.no_grad():
    outputs = model(**inputs)  # Forward pass through the model to get embeddings


In [ ]:
!pip install pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 6.4 MB/s eta 0:00:00


In [ ]:
!pip install -qU \
  pinecone==6.0.2 \
  pinecone-datasets==1.0.2 \
  sentence-transformers==3.4.1 \
  pinecone-notebooks==0.1.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 769.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 76.0 MB/s eta 0:00:00


In [ ]:
import pinecone
pinecone = pinecone.Pinecone(api_key="pcsk_5YXa8U_DyeZzB7q5nEzZXycR1Ec3QC8bKbFgNMaK6guyFWUobEte7Z4PAXjvzMoBgZXQ7N", environment="us-west1-gcp")

In [ ]:
from pinecone_datasets import load_dataset

dataset = load_dataset('quora_all-MiniLM-L6-bm25')

# The metadata we need is actually stored in the "blob" column so let's rename it
dataset.documents.drop(['metadata'], axis=1, inplace=True)
dataset.documents.rename(columns={'blob': 'metadata'}, inplace=True)

# We don't need sparse_values for this demo either so let's drop those as well
dataset.documents.drop(['sparse_values'], axis=1, inplace=True)

# To speed things up in this demo, we will use 80K rows of the dataset between rows 240K -> 320K
dataset.documents.drop(dataset.documents.index[320_000:], inplace=True)
dataset.documents.drop(dataset.documents.index[:240_000], inplace=True)
dataset.head()

Loading documents parquet files:   0%|          | 0/10 [00:00<?, ?it/s]

,id,values,metadata
240000,515997,"[-0.00531694, 0.06937869, -0.0092854, 0.003286...","{'text': ' Why is a ""law of sciences"" importan..."
240001,515998,"[-0.09243751, 0.065432355, -0.06946959, 0.0669...",{'text': ' Is it possible to format a BitLocke...
240002,515999,"[-0.021924071, 0.032280188, -0.020190848, 0.07...",{'text': ' Can formatting a hard drive stress ...
240003,516000,"[-0.120020054, 0.024080949, 0.10693012, -0.018...",{'text': ' Are the new Samsung Galaxy J7 and J...
240004,516001,"[-0.095293395, -0.048446465, -0.017618902, -0....",{'text': ' I just watched an add for Indonesia...


In [ ]:
print(f"Rows in dataset: {len(dataset)}")

Rows in dataset: 80000


In [ ]:
row1 = dataset.documents.iloc[0:1].to_dict(orient="records")[0]
dimension = len(row1['values'])
print(f"These embeddings have dimension {dimension}")

These embeddings have dimension 384


In [ ]:
print("Here are some example questions in the data set:\n")
for r in dataset.documents.iloc[0:10].to_dict(orient="records"):
    print("  -" + r['metadata']['text'])

Here are some example questions in the data set:

  - Why is a "law of sciences" important for our life?
  - Is it possible to format a BitLocker or FileVault protected drive?
  - Can formatting a hard drive stress it out?
  - Are the new Samsung Galaxy J7 and J5 worth their price?
  - I just watched an add for Indonesia 2026 World Cup bid in YouTube, is it viable?
  - I am an 18 year old college student. Is it a viable idea to play poker in order to pay for my college tuition?
  - If the French monarchy had never been abolished, who would be the current king/queen?
  - Who was the best French King?
  - How do I obtain a free United States phone number using the Internet?
  - What is the change in your opinion about PM Narendra Modi after demonetization of 1000 and 500 rupees currency notes?


In [ ]:
import os

if not os.environ.get("your-api-key"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()

In [ ]:
from pinecone import Pinecone

# Initialize client
pc = Pinecone(api_key=os.environ.get("your-api-key"))

In [ ]:
from pinecone import ServerlessSpec

index_name = 'semantic-search-fast'

# Check if index already exists (it shouldn't if this is first time running the demo)
if not pc.has_index(name=index_name):
    # If does not exist, create index
    pc.create_index(
        name=index_name,
        dimension=384, # dimensionality of MiniLM
        metric='dotproduct',
        spec = ServerlessSpec(
            cloud='aws',
            region='us-east-1'
        )
    )

# Initialize index client
index = pc.Index(name=index_name)

# View index stats
index.describe_index_stats()

{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'': {'vector_count': 80000}},
 'total_vector_count': 80000,
 'vector_type': 'dense'}

In [ ]:
from tqdm import tqdm

batch_size = 100

for start in tqdm(range(0, len(dataset.documents), batch_size), "Upserting records batch"):
    batch = dataset.documents.iloc[start:start + batch_size].to_dict(orient="records")
    index.upsert(vectors=batch)

Upserting records batch: 100%|██████████| 800/800 [05:27<00:00,  2.45it/s]


In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)
model

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [ ]:
def find_similar_questions(question):
    # Embed the question into a query vector
    xq = model.encode(question).tolist()

    # Now query Pinecone to find similar questions
    return index.query(vector=xq, top_k=5, include_metadata=True)


In [ ]:
question = "Which city has the highest population in the world?"
xc = find_similar_questions(question)
xc

{'matches': [{'id': '69331',
              'metadata': {'text': " What's the world's largest city?"},
              'score': 0.785789371,
              'values': []},
             {'id': '69332',
              'metadata': {'text': ' What is the biggest city?'},
              'score': 0.727474093,
              'values': []},
             {'id': '84749',
              'metadata': {'text': " What are the world's most advanced "
                                   'cities?'},
              'score': 0.709189713,
              'values': []},
             {'id': '109231',
              'metadata': {'text': ' Where is the most beautiful city in the '
                                   'world?'},
              'score': 0.696055055,
              'values': []},
             {'id': '109230',
              'metadata': {'text': ' What is the greatest, most beautiful city '
                                   'in the world?'},
              'score': 0.657444656,
              'values': []}],
 'namesp

In [ ]:
def print_query_results(results):
    for result in results['matches']:
        print(f"{round(result['score'], 2)}: {result['metadata']['text']}")

print_query_results(xc)

0.79:  What's the world's largest city?
0.73:  What is the biggest city?
0.71:  What are the world's most advanced cities?
0.7:  Where is the most beautiful city in the world?
0.66:  What is the greatest, most beautiful city in the world?


In [ ]:
question2 = "Which metropolis has the highest number of people?"

xc2 = find_similar_questions(question2)
print_query_results(xc2)

0.64:  What is the biggest city?
0.6:  What is the most dangerous city in USA?
0.59:  What's the world's largest city?
0.59:  What is the most dangerous city in USA? Why?
0.58:  What are the world's most advanced cities?
